In [5]:
import pandas as pd
from explainability_utils import *
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import h5py

sys.path.append(os.path.abspath("../training"))
from training_utils import CNN2D, SpectraDataset_s, SpectraDataset_p

In [6]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
pl.seed_everything(SEED)

Seed set to 42


42

In [7]:
output_path = "temporal_shap"
os.makedirs(output_path, exist_ok=True)
MAX_EVALS = 5000
base_path = r'..\preprocessed_dset_1'
f_t_range_64_path = os.path.join(base_path, 'sp_64', "f_t_range.npy")
tf64 = np.load(f_t_range_64_path)

ft = [tf64[:2], tf64[2:]]

# Setting for the datasets
mean_64, std_64 = spectra_stats(os.path.join(base_path, 'sp_64', "train"))

transform_64 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean_64, std=std_64)
])

# Load the features and metadata CSV files
df = pd.read_csv(r'..\preprocessed_dset_1\features_and_metadata.csv')
test_df = df[df["split"] == "test"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

(..\preprocessed_dset_1\sp_64\train) Mean: [0.67176665 0.65817659 0.6412554 ], Std: [0.1186761  0.12021562 0.11939131] (calculated and saved)
Using device: cuda


#### Week division

In [5]:
# Convert the trace_start_time column to datetime (if not already)
test_df['trace_start_time'] = pd.to_datetime(test_df['trace_start_time'])

# Get the ISO week number for each event
test_df['week'] = test_df['trace_start_time'].dt.isocalendar().week - 34 # Riscaliamo per far partire la prima settimana da 0

# Group by week number and collect trace names
week_dict = test_df.groupby('week')['trace_name'].apply(list).to_dict()
week_dict.keys()

C:\Users\frmar\AppData\Local\Temp\ipykernel_22468\4047119665.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['trace_start_time'] = pd.to_datetime(test_df['trace_start_time'])
C:\Users\frmar\AppData\Local\Temp\ipykernel_22468\4047119665.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['week'] = test_df['trace_start_time'].dt.isocalendar().week - 34 # Riscaliamo per far partire la prima settimana da 0


dict_keys([np.uint32(0), np.uint32(1), np.uint32(2), np.uint32(3), np.uint32(4), np.uint32(5), np.uint32(6), np.uint32(7), np.uint32(8), np.uint32(9), np.uint32(10), np.uint32(11), np.uint32(12), np.uint32(13), np.uint32(14), np.uint32(15), np.uint32(16), np.uint32(17)])

In [6]:
first_days = test_df.groupby('week')['trace_start_time'].min().apply(
    lambda dt: (dt - pd.Timedelta(days=dt.weekday())).date()
)
for offset_week, first_day in first_days.sort_index().items():
    print(f"Offset week {offset_week} (ISO week {offset_week + 34}) - First day: {first_day}")

Offset week 0 (ISO week 34) - First day: 2016-08-22
Offset week 1 (ISO week 35) - First day: 2016-08-29
Offset week 2 (ISO week 36) - First day: 2016-09-05
Offset week 3 (ISO week 37) - First day: 2016-09-12
Offset week 4 (ISO week 38) - First day: 2016-09-19
Offset week 5 (ISO week 39) - First day: 2016-09-26
Offset week 6 (ISO week 40) - First day: 2016-10-03
Offset week 7 (ISO week 41) - First day: 2016-10-10
Offset week 8 (ISO week 42) - First day: 2016-10-17
Offset week 9 (ISO week 43) - First day: 2016-10-24
Offset week 10 (ISO week 44) - First day: 2016-10-31
Offset week 11 (ISO week 45) - First day: 2016-11-07
Offset week 12 (ISO week 46) - First day: 2016-11-14
Offset week 13 (ISO week 47) - First day: 2016-11-21
Offset week 14 (ISO week 48) - First day: 2016-11-28
Offset week 15 (ISO week 49) - First day: 2016-12-05
Offset week 16 (ISO week 50) - First day: 2016-12-12
Offset week 17 (ISO week 51) - First day: 2016-12-19


In [7]:
# Check if every value in week 10 onwards is labeled 'post'
week_10_onwards = test_df[test_df['week'] >= 10]
week_10_post_check = (week_10_onwards['label'] == 'post').all()

# Check if every value in week 9 and before is labeled 'pre'
week_9_before = test_df[test_df['week'] <= 9]
week_9_pre_check = (week_9_before['label'] == 'pre').all()

print(f"All values in week 10 onwards are labeled 'post': {week_10_post_check}")
print(f"All values in week 9 and before are labeled 'pre': {week_9_pre_check}")

All values in week 10 onwards are labeled 'post': True
All values in week 9 and before are labeled 'pre': True


In [20]:
import os
import shutil

# Define the source and destination directories
source_dir = r"..\preprocessed_dset_1\sp_64\test"
destination_dir = r"..\preprocessed_dset_1\sp_64\weeks_split"

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Iterate over the week_dict to create folders and move files
for week, trace_names in week_dict.items():
    week_folder = os.path.join(destination_dir, f'{week}')
    os.makedirs(week_folder, exist_ok=True)
    
    for trace_name in trace_names:
        label = 'pre' if week <= 9 else 'post'
        file_name = f"{trace_name}_{label}.png"
        source_file = os.path.join(source_dir, file_name)
        destination_file = os.path.join(week_folder, file_name)
        
        if os.path.exists(source_file):
            shutil.copy(source_file, destination_file)
        else:
            print(f"File {source_file} does not exist.")

#### SHAP Tensors P

In [10]:
best_model_p_64 = "../training/models/p_wave_model_checkpoints/model_checkpoints_64_08/best_model_fold_s.ckpt"
dim_p64 = (33, 188)

model_p = CNN2D.load_from_checkpoint(best_model_p_64, input_dim=dim_p64, num_classes=2)
model_p.eval()
model_p.to(device)
print("Model loaded")

Model loaded


In [ ]:
to_take = 5

for i in range(len(week_dict.keys())):
    img_path = rf"..\preprocessed_dset_1\sp_64\weeks_split\{i}"
    dset = SpectraDataset_p(img_path, transform=transform_64, get_image_name=True)
    dloader = DataLoader(dset, batch_size=1, shuffle=False)
    took = 0
    for sample in dloader:
        img, label, name = sample
        img = img.to(device)
        label = label.to(device)
        pred = model_p(img)
        pred = torch.argmax(pred, dim=1)
        if pred == label:
            took += 1
            np.save(os.path.join(output_path, f"p_{i}_{name[0]}_img.npy"), img.cpu().numpy())
            shap_tensor = compute_shap_tensor(model_p, sample, dim_p64, device, max_evals = MAX_EVALS, masker_settings="inpaint_telea", save_path = os.path.join(output_path, f"p_{i}_{name[0]}.png")) 
            if took == to_take:
                break

#### Final Plots P

In [14]:
wf_path = '../dset/data/catalogs'
shap_path = r'temporal_shap'
file_names = [f for f in os.listdir(shap_path) if (f.endswith('EV.png.npy') and f.startswith('p'))]
os.makedirs("temporal_shap/plots", exist_ok=True)
alpha_min, alpha_max = np.inf, -np.inf

"""for f in file_names:
    shap_tensor = np.load(os.path.join(shap_path, f))
    alpha_min = min(alpha_min, shap_tensor.min())
    alpha_max = max(alpha_max, shap_tensor.max())"""

for file in tqdm(file_names):
    week = int(file.split('_')[1])
    name = (file.split('_')[2]+'_'+file.split('_')[3])[:-8]
    label = 'pre' if week <= 9 else 'post'
    
    hdf = h5py.File(f'{wf_path}/NRCA/NRCA_{label}.hdf5', 'r')
    waveform = hdf[name][:]

    img = np.load(os.path.join(shap_path, f"p_{week}_{name}_img.npy")).squeeze(axis=0)
    shap_tensor = np.load(os.path.join(shap_path, f"p_{week}_{name}.png.npy")).squeeze(axis=0)
    alt_wave = df[df['trace_name'] == name].p_s_diff_sec.values[0] + 5
    day = df[df['trace_name'] == name].trace_start_time.values[0] 

    plot_wf_shap(shap_tensor = shap_tensor, 
        spectrogram = img, 
        waveform = waveform, 
        alt_wave = alt_wave,
        ft = ft,
        alpha_min = None,
        alpha_max = None,
        name = name,
        label = label,
        day = day,
        week = week,
        spec_type = "p64_08",
        show = False,
        save_path = f"temporal_shap/plots/p_{week}_{name}.png",
        figsize = (15, 8))
    
    hdf.close()


100%|██████████| 90/90 [02:55<00:00,  1.95s/it]


#### SHAP Tensors S

In [ ]:
best_model_s_64 = "../training/models/s_wave_model_checkpoints/model_checkpoints_64_08/best_model_fold_s.ckpt"
dim_s64 = (33, 150)

model_s = CNN2D.load_from_checkpoint(best_model_s_64, input_dim=dim_s64, num_classes=2)
model_s.eval()
model_s.to(device)
print("Model loaded")

In [ ]:
to_take = 5

for i in range(len(week_dict.keys())):
    img_path = rf"..\preprocessed_dset_1\sp_64\weeks_split\{i}"
    dset = SpectraDataset_s(img_path, transform=transform_64,meta_path=r"..\preprocessed_dset_1\features_and_metadata.csv", get_image_name=True)
    dloader = DataLoader(dset, batch_size=1, shuffle=False)
    took = 0
    for sample in dloader:
        img, label, name = sample
        img = img.to(device)
        label = label.to(device)
        pred = model_s(img)
        pred = torch.argmax(pred, dim=1)
        if pred == label:
            took += 1
            np.save(os.path.join(output_path, f"s_{i}_{name[0]}_img.npy"), img.cpu().numpy())
            shap_tensor = compute_shap_tensor(model_s, sample, dim_s64, device, max_evals = MAX_EVALS, masker_settings="inpaint_telea", save_path = os.path.join(output_path, f"s_{i}_{name[0]}.png")) 
            if took == to_take:
                break

#### Final Plots S

In [ ]:
wf_path = '../dset/data/catalogs'
shap_path = r'temporal_shap'
file_names = [f for f in os.listdir(shap_path) if (f.endswith('EV.png.npy') and f.startswith('s'))]
os.makedirs("temporal_shap/plots", exist_ok=True)
alpha_min, alpha_max = np.inf, -np.inf

"""for f in file_names:
    shap_tensor = np.load(os.path.join(shap_path, f))
    alpha_min = min(alpha_min, shap_tensor.min())
    alpha_max = max(alpha_max, shap_tensor.max())"""

for file in tqdm(file_names):
    week = int(file.split('_')[1])
    name = (file.split('_')[2]+'_'+file.split('_')[3])[:-8]
    label = 'pre' if week <= 9 else 'post'
    
    hdf = h5py.File(f'{wf_path}/NRCA/NRCA_{label}.hdf5', 'r')
    waveform = hdf[name][:]

    img = np.load(os.path.join(shap_path, f"s_{week}_{name}_img.npy")).squeeze(axis=0)
    shap_tensor = np.load(os.path.join(shap_path, f"s_{week}_{name}.png.npy")).squeeze(axis=0)
    alt_wave = 5 - df[df['trace_name'] == name].p_s_diff_sec.values[0]
    day = df[df['trace_name'] == name].trace_start_time.values[0] 

    plot_wf_shap(shap_tensor = shap_tensor, 
        spectrogram = img, 
        waveform = waveform, 
        alt_wave = alt_wave,
        ft = ft,
        alpha_min = None,
        alpha_max = None,
        name = name,
        label = label,
        day = day,
        week = week,
        spec_type = "s64_08",
        show = False,
        save_path = f"temporal_shap/plots/s_{week}_{name}.png",
        figsize = (15, 8))
    
    hdf.close()